In [24]:
import pandas as pd
import re

# Load O*NET clean file
onet = pd.read_csv("C:\\Users\\Lamees\\Documents\\Lamees Group project\\ONet framework files\\ONET_files.csv", dtype=str)

# Tidy function (same as before for consistency)
def tidy(text):
    t = str(text).lower()
    t = re.sub(r"[^a-z ]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

# Build O*NET pool
onet_pool = onet[["O*NET-SOC Code", "Title"]].dropna()
onet_pool = onet_pool.rename(columns={"O*NET-SOC Code":"onet_code", "Title":"title"})
onet_pool["label"] = onet_pool["title"].apply(tidy)
onet_pool = onet_pool.drop_duplicates().reset_index(drop=True)

print("O*NET pool size:", len(onet_pool))


O*NET pool size: 923


In [25]:
# Load ESCO + ISCO pools
isco = pd.read_csv("C:\\Users\\Lamees\\Documents\\Lamees Group project\\needed\\needed\\1. cleaned framework datasets\\isco_clean.csv", dtype=str)
occ = pd.read_csv("C:\\Users\\Lamees\\Documents\\Lamees Group project\\needed\\needed\\1. cleaned framework datasets\\esco_occupations_clean.csv", dtype=str)

pool1 = isco.rename(columns={"isco_title": "label"})[["isco_code", "label"]]
pool2 = occ.rename(columns={"occupation": "label"})[["isco_code", "label"]]
alt = occ[["isco_code", "altLabels"]].dropna()
alt["label"] = alt["altLabels"].str.split("\n")
alt = alt.explode("label")[["isco_code", "label"]]

esco_pool = pd.concat([pool1, pool2, alt]).dropna()
esco_pool["label"] = esco_pool["label"].apply(tidy)
esco_pool = esco_pool.drop_duplicates().reset_index(drop=True)

# Combine all three frameworks
framework_pool = pd.concat([
    esco_pool.assign(source="ESCO"),
    onet_pool.assign(source="ONET"),
    isco.rename(columns={"isco_title":"label"})[["isco_code","label"]].assign(source="ISCO")
])

framework_pool["label"] = framework_pool["label"].apply(tidy)
framework_pool = framework_pool.drop_duplicates().reset_index(drop=True)

print("Unified pool size:", len(framework_pool))


Unified pool size: 47245


In [4]:
import pandas as pd
import re

# --- Load framework pools ---
isco = pd.read_csv("C:\\Users\\Lamees\\Documents\\Lamees Group project\\needed\\needed\\1. cleaned framework datasets\\isco_clean.csv", dtype=str)
occ = pd.read_csv("C:\\Users\\Lamees\\Documents\\Lamees Group project\\needed\\needed\\1. cleaned framework datasets\\esco_occupations_clean.csv", dtype=str)
onet = pd.read_csv("C:\\Users\\Lamees\\Documents\\Lamees Group project\\ONet framework files\\ONET_files.csv", dtype=str)

# --- Build pools ---
pool1 = isco.rename(columns={"isco_title": "label"})[["isco_code", "label"]]
pool2 = occ.rename(columns={"occupation": "label"})[["isco_code", "label"]]
alt = occ[["isco_code", "altLabels"]].dropna()
alt["label"] = alt["altLabels"].str.split("\n")
alt = alt.explode("label")[["isco_code", "label"]]
esco_pool = pd.concat([pool1, pool2, alt]).dropna()

onet_pool = onet[["O*NET-SOC Code", "Title"]].dropna()
onet_pool = onet_pool.rename(columns={"O*NET-SOC Code":"onet_code", "Title":"label"})


# --- Tidy function ---
remove_words = [
    "senior", "sr", "junior", "jr", "lead", "principal", "staff",
    "associate", "intern", "trainee", "head of", "chief", "ii", "iii", "iv"
]

def tidy(title):
    t = str(title).lower()

    # Remove text inside parentheses and square brackets
    t = re.sub(r"\(.*?\)|\[.*?\]", " ", t)

    # Keep only the part before common separators
    t = re.split(r"\s+-\s+|\s+–\s+|\s+—\s+|\s+\|\s+|,|/", t)[0]

    # Keep letters and spaces only
    t = re.sub(r"[^a-z ]", " ", t)

    # Remove seniority words
    for w in remove_words:
        t = re.sub(r"\b" + re.escape(w) + r"\b", " ", t)

    # Remove extra spaces
    t = re.sub(r"\s+", " ", t).strip()

    return t


# --- Apply tidy ---
esco_pool["label"] = esco_pool["label"].apply(tidy)
esco_pool = esco_pool[esco_pool["label"] != ""].drop_duplicates("label").reset_index(drop=True)

onet_pool["label"] = onet_pool["label"].apply(tidy)
onet_pool = onet_pool[onet_pool["label"] != ""].drop_duplicates("label").reset_index(drop=True)

# --- Load and tidy job titles ---
jobs = pd.read_csv("C:\\Users\\Lamees\\Documents\\Lamees Group project\\ONET+framework+llm\\biggggg_sample.csv")
jobs["title_clean"] = jobs["title"].apply(tidy)
jobs = jobs[jobs["title_clean"] != ""]
counts = jobs["title_clean"].value_counts()

titles = pd.DataFrame({"title_clean": counts.index, "postings": counts.values})

# --- Exact match against ESCO/ISCO ---
exact_esco = titles.merge(esco_pool, left_on="title_clean", right_on="label", how="left")
matched_esco = exact_esco[exact_esco["isco_code"].notna()]

# --- Exact match against O*NET ---
exact_onet = titles.merge(onet_pool, left_on="title_clean", right_on="label", how="left")
matched_onet = exact_onet[exact_onet["onet_code"].notna()]

# --- Coverage stats ---
total_postings = counts.sum()
esco_coverage = round(matched_esco["postings"].sum() / total_postings * 100, 1)
onet_coverage = round(matched_onet["postings"].sum() / total_postings * 100, 1)

print("Exact matched titles (ESCO/ISCO):", len(matched_esco))
print("Postings covered by ESCO/ISCO exact matches:", esco_coverage, "%")
print("Exact matched titles (O*NET):", len(matched_onet))
print("Postings covered by O*NET exact matches:", onet_coverage, "%")

# --- Save tidied titles for fuzzy/LLM step ---
titles.to_csv("titles_tidied3.csv", index=False)


Exact matched titles (ESCO/ISCO): 1107
Postings covered by ESCO/ISCO exact matches: 39.8 %
Exact matched titles (O*NET): 18
Postings covered by O*NET exact matches: 0.1 %


In [5]:
# --- Save exact matches for later merging ---
# ESCO/ISCO exact matches
matched_esco_out = matched_esco[["title_clean", "isco_code", "label"]].copy()
matched_esco_out = matched_esco_out.rename(columns={"isco_code":"code"})
matched_esco_out["framework"] = "ESCO/ISCO"
matched_esco_out["method"] = "exact"
matched_esco_out["mapped_label"] = matched_esco_out["label"]

# O*NET exact matches
matched_onet_out = matched_onet[["title_clean", "onet_code", "label"]].copy()
matched_onet_out = matched_onet_out.rename(columns={"onet_code":"code"})
matched_onet_out["framework"] = "ONET"
matched_onet_out["method"] = "exact"
matched_onet_out["mapped_label"] = matched_onet_out["label"]

# Combine both into one file
titles_exact_matched2 = pd.concat([matched_esco_out, matched_onet_out], ignore_index=True)
titles_exact_matched2.to_csv("titles_exact_matched2.csv", index=False)

print("Saved exact matches to titles_exact_matched2.csv")
print(titles_exact_matched2.head(20))


Saved exact matches to titles_exact_matched2.csv
                      title_clean  code                          label  \
0                    data analyst  2511                   data analyst   
1                   data engineer  2511                  data engineer   
2                  data scientist  2511                 data scientist   
3               software engineer  2512              software engineer   
4       machine learning engineer  2511      machine learning engineer   
5                     ai engineer  3240                    ai engineer   
6                business analyst  2421               business analyst   
7        computer vision engineer  2511       computer vision engineer   
8                         manager  1120                        manager   
9                   ai specialist  3240                  ai specialist   
10                product manager  1223                product manager   
11                       director  1120                       d

In [6]:
from rapidfuzz import process, fuzz
import ollama

MODEL = "qwen2.5:7b"

# Candidate search restricted by domain keywords
DOMAIN_KEYWORDS = ["data","software","ai","ml","cloud","engineer","developer","scientist","analyst"]

def get_candidates(title):
    results = process.extract(title, framework_pool["label"].tolist(), scorer=fuzz.token_sort_ratio, limit=50)
    out = []
    for label, score, i in results:
        # Domain filter: only keep candidates relevant to IT/data
        if any(word in label for word in DOMAIN_KEYWORDS):
            out.append((framework_pool.iloc[i]["source"], framework_pool.iloc[i].get("isco_code", framework_pool.iloc[i].get("onet_code","")), label))
    return out[:10]

def ask_llm(title, cands):
    options = "\n".join([f"{n+1}. {c[2]} ({c[0]})" for n,c in enumerate(cands)])
    prompt = f"""Job title: {title}

Candidate occupations:
{options}

Choose the candidate that best matches the job title. Reply ONLY with the number, or NONE if none fit. Do not guess."""
    reply = ollama.chat(model=MODEL,
                        messages=[{"role":"user","content":prompt}],
                        options={"temperature":0})
    answer = reply["message"]["content"].strip()
    if answer.isdigit() and 1 <= int(answer) <= len(cands):
        return cands[int(answer)-1]
    return ("NONE","NONE","NONE")


In [7]:
titles = pd.read_csv("titles_tidied3.csv") 

#Load exact matches and exclude them
exact = pd.read_csv("titles_exact_matched2.csv")
titles_for_llm = titles[~titles["title_clean"].isin(exact["title_clean"])]

#Limit to top 1000 for testing
titles_for_llm = titles_for_llm.head(1000)    # test on top 1000

results = []
for title in titles_for_llm["title_clean"]:
    cands = get_candidates(title)
    source, code, label = ask_llm(title, cands)
    results.append({"title_clean":title,"framework":source,"code":code,"mapped_label":label})

df = pd.DataFrame(results)
df.to_csv("title_to_framework2.csv", index=False)
print(df.head(20))

# Count how many jobs were not mapped
none_count = (df["framework"] == "NONE").sum()
total_count = len(df)

print(f"Total job titles: {total_count}")
print(f"Unmapped (NONE): {none_count}")
print(f"Mapped: {total_count - none_count}")
print(f"Percentage unmapped: {round(none_count/total_count*100, 1)}%")

                             title_clean framework  code  \
0                                     ai      NONE  NONE   
1                        prompt engineer      ESCO  2149   
2                           nlp engineer      NONE  NONE   
3                     ai product manager      NONE  NONE   
4                          ai researcher      ESCO  2149   
5                          ai consultant      NONE  NONE   
6                 deep learning engineer      NONE  NONE   
7   machine learning operations engineer      ESCO  2511   
8                      account executive      NONE  NONE   
9                       registered nurse      NONE  NONE   
10                      backend engineer      NONE  NONE   
11          enterprise account executive      NONE  NONE   
12      sales development representative      NONE  NONE   
13                   azure data engineer      ESCO  2511   
14                   full stack engineer      NONE  NONE   
15     digital training facility manager

In [8]:
import pandas as pd

# Load tidied titles with posting counts
titles = pd.read_csv("titles_tidied3.csv")

# Load exact matches and attach postings
exact = pd.read_csv("titles_exact_matched2.csv")
exact_with_counts = exact.merge(titles, on="title_clean", how="left")

# Load LLM matches and attach postings
llm = pd.read_csv("title_to_framework2.csv")
llm_with_counts = llm.merge(titles, on="title_clean", how="left")

# Combine both into one dataset
combined = pd.concat([exact_with_counts, llm_with_counts], ignore_index=True)

# Save unified dataset
combined.to_csv("job_title_results.csv", index=False)

print("Saved combined results to job_title_results.csv")
print(combined.head(20))

# Stats
total_postings = titles["postings"].sum()
mapped_postings = combined[combined["framework"] != "NONE"]["postings"].sum()
unmapped_postings = total_postings - mapped_postings

print(f"Total postings: {total_postings}")
print(f"Mapped postings: {mapped_postings} ({round(mapped_postings/total_postings*100,1)}%)")
print(f"Unmapped postings: {unmapped_postings} ({round(unmapped_postings/total_postings*100,1)}%)")

# Breakdown by method
exact_postings = exact_with_counts["postings"].sum()
llm_postings = llm_with_counts[llm_with_counts["framework"] != "NONE"]["postings"].sum()

print(f"Postings covered by exact matches: {exact_postings} ({round(exact_postings/total_postings*100,1)}%)")
print(f"Postings covered by LLM matches: {llm_postings} ({round(llm_postings/total_postings*100,1)}%)")


Saved combined results to job_title_results.csv
                      title_clean  code                          label  \
0                    data analyst  2511                   data analyst   
1                   data engineer  2511                  data engineer   
2                  data scientist  2511                 data scientist   
3               software engineer  2512              software engineer   
4       machine learning engineer  2511      machine learning engineer   
5                     ai engineer  3240                    ai engineer   
6                business analyst  2421               business analyst   
7        computer vision engineer  2511       computer vision engineer   
8                         manager  1120                        manager   
9                   ai specialist  3240                  ai specialist   
10                product manager  1223                product manager   
11                       director  1120                       di

In [9]:
import pandas as pd

# Load combined dataset
combined = pd.read_csv("job_title_results.csv")

# Fill in blank/null values in 'method' with 'llm'
combined["method"] = combined["method"].fillna("llm")

# Save back to file
combined.to_csv("job_title_results.csv", index=False)

print(combined["method"].value_counts())


method
exact    1125
llm      1000
Name: count, dtype: int64


In [10]:
# To ensure all onet codes appear on the csv file
import pandas as pd
import re

# Load combined results
combined = pd.read_csv("job_title_results.csv")

# Load O*NET pool
onet = pd.read_csv("C:\\Users\\Lamees\\Documents\\Lamees Group project\\ONet framework files\\ONET_files.csv", dtype=str)

# Tidy function
remove_words = [
    "senior", "sr", "junior", "jr", "lead", "principal", "staff",
    "associate", "intern", "trainee", "head of", "chief", "ii", "iii", "iv"
]

def tidy(title):
    t = str(title).lower()

    # Remove text inside parentheses and square brackets
    t = re.sub(r"\(.*?\)|\[.*?\]", " ", t)

    # Keep only the part before common separators
    t = re.split(r"\s+-\s+|\s+–\s+|\s+—\s+|\s+\|\s+|,|/", t)[0]

    # Keep letters and spaces only
    t = re.sub(r"[^a-z ]", " ", t)

    # Remove seniority words
    for w in remove_words:
        t = re.sub(r"\b" + re.escape(w) + r"\b", " ", t)

    # Remove extra spaces
    t = re.sub(r"\s+", " ", t).strip()

    return t

# Prepare O*NET pool
onet["label"] = onet["Title"].apply(tidy)
onet_pool = onet.dropna(subset=["label"]).drop_duplicates("label")[["label","O*NET-SOC Code"]]

# Merge to fill missing codes
combined = combined.merge(onet_pool, left_on="mapped_label", right_on="label", how="left")

# Fill in blanks: if framework is ONET and code is NONE/blank, use the O*NET-SOC Code
combined.loc[(combined["framework"]=="ONET") & ((combined["code"].isna()) | (combined["code"]=="NONE")), "code"] = combined["O*NET-SOC Code"]

# Drop helper columns only if they exist
for col in ["label_x", "label_y", "O*NET-SOC Code"]:
    if col in combined.columns:
        combined = combined.drop(columns=[col])

# Save back
combined.to_csv("job_title_results_filled.csv", index=False)

print("Patched codes saved to job_title_results_filled.csv")
print(combined.head(10))


Patched codes saved to job_title_results_filled.csv
                 title_clean  code  framework method  \
0               data analyst  2511  ESCO/ISCO  exact   
1              data engineer  2511  ESCO/ISCO  exact   
2             data scientist  2511  ESCO/ISCO  exact   
3          software engineer  2512  ESCO/ISCO  exact   
4  machine learning engineer  2511  ESCO/ISCO  exact   
5                ai engineer  3240  ESCO/ISCO  exact   
6           business analyst  2421  ESCO/ISCO  exact   
7   computer vision engineer  2511  ESCO/ISCO  exact   
8                    manager  1120  ESCO/ISCO  exact   
9              ai specialist  3240  ESCO/ISCO  exact   

                mapped_label  postings  
0               data analyst      3380  
1              data engineer      3030  
2             data scientist      2505  
3          software engineer      1961  
4  machine learning engineer       942  
5                ai engineer       872  
6           business analyst       566  
7  

In [11]:
#Skills 
import pandas as pd
import re

# Tidy function for skills
def tidy_skill(s):
    s = str(s).lower()
    s = re.sub(r"[-_]", " ", s)           # normalize hyphens/underscores
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Load ESCO skills
esco_sk = pd.read_csv("C:\\Users\\Lamees\\Documents\\Lamees Group project\\needed\\needed\\1. cleaned framework datasets\\esco_skills_clean.csv", dtype=str)

# ESCO pool
p1 = esco_sk[["skill_uri","skill"]].rename(columns={"skill":"label"})
p2 = esco_sk[["skill_uri","altLabels"]].dropna()
p2["label"] = p2["altLabels"].str.split("\n")
p2 = p2.explode("label")[["skill_uri","label"]]
esco_pool = pd.concat([p1,p2]).dropna()
esco_pool["label"] = esco_pool["label"].apply(tidy_skill)
esco_pool = esco_pool[esco_pool["label"]!=""].drop_duplicates().assign(framework="ESCO")

# Load O*NET skills (Element Name column)
onet = pd.read_csv("C:\\Users\\Lamees\\Documents\\Lamees Group project\\ONet framework files\\ONET_files.csv", dtype=str)
onet_pool = onet[["Element ID","Element Name"]].dropna()
onet_pool = onet_pool.rename(columns={"Element ID":"onet_code","Element Name":"label"})
onet_pool["label"] = onet_pool["label"].apply(tidy_skill)
onet_pool = onet_pool[onet_pool["label"]!=""].drop_duplicates().assign(framework="ONET")

# Unified skill pool
skill_pool = pd.concat([esco_pool, onet_pool]).drop_duplicates().reset_index(drop=True)
print("Unified skill pool size:", len(skill_pool))


Unified skill pool size: 109070


In [12]:
# Load jobs dataset for skills
jobs = pd.read_csv("C:\\Users\\Lamees\\Documents\\Lamees Group project\\ONET+framework+llm\\biggggg_sample.csv")
jobs = jobs.dropna(subset=["skills"])

# Split skills column into lists
def split_skills(text):
    text = str(text)
    if "'" in text:  # looks like ['python','sql']
        found = re.findall(r"'([^']*)'", text)
    else:            # looks like Python, SQL, AWS
        found = text.strip("[]").split(",")
    return [tidy_skill(x) for x in found if x.strip()!=""]

jobs["skill_list"] = jobs["skills"].apply(split_skills)

# Count skills
all_skills = jobs["skill_list"].explode().dropna()
skill_counts = all_skills.value_counts()
print("Unique skills in postings:", len(skill_counts))


Unique skills in postings: 4886


In [13]:
# Build table of skills
table = pd.DataFrame({"skill_clean":skill_counts.index,"mentions":skill_counts.values})

# Exact match against unified pool
exact = table.merge(skill_pool, left_on="skill_clean", right_on="label", how="left")

matched = exact[exact["framework"].notna()]
unmatched = exact[exact["framework"].isna()]

# Total mentions across all postings
total_mentions = all_skills.shape[0]

# Deduplicate matched skills to avoid double-counting
matched_unique = matched.drop_duplicates(subset=["skill_clean"])

# Coverage calculation
coverage = round(matched_unique["mentions"].sum() / total_mentions * 100, 1)

print("Exact matched skills:", len(matched_unique))
print("Mentions covered by exact matches:", coverage, "%")

# Save to CSV
table.to_csv("skills_table.csv", index=False)

print("Skills table saved to skills_table.csv")
print(table.head(10))


Exact matched skills: 698
Mentions covered by exact matches: 50.8 %
Skills table saved to skills_table.csv
          skill_clean  mentions
0                 sql     14995
1              python     14596
2               excel      9367
3       communication      7112
4                 aws      6750
5               azure      5629
6  project management      5598
7                   r      4810
8                java      4635
9               spark      4424


In [14]:
# Build table of skills from postings
table = pd.DataFrame({"skill_clean": skill_counts.index, "mentions": skill_counts.values})

# Exact match against unified pool
exact = table.merge(skill_pool, left_on="skill_clean", right_on="label", how="left")

# Add mapped_label column
exact["mapped_label"] = exact["label"]

# Keep only matched rows
matched = exact[exact["framework"].notna()]

# Add missing columns
matched = matched.assign(
    code=matched["skill_uri"].fillna(matched["onet_code"]),
    method="exact"
)

# Final exact match dataset
skill_exact_match = matched[["skill_clean", "framework", "code", "method", "mapped_label"]]

# Preview
print(skill_exact_match.head(10))

# Save exact matches for skills
skill_exact_match.to_csv("skill_exact_match.csv", index=False)


           skill_clean framework  \
0                  sql      ESCO   
1                  sql      ESCO   
2               python      ESCO   
3               python      ONET   
5        communication      ESCO   
8   project management      ESCO   
9   project management      ESCO   
10                   r      ESCO   
11                   r      ONET   
12                java      ESCO   

                                                 code method  \
0   http://data.europa.eu/esco/skill/598de5b0-5b58...  exact   
1   http://data.europa.eu/esco/skill/ab1e97ed-2319...  exact   
2   http://data.europa.eu/esco/skill/ccd0a1d9-afda...  exact   
3                                             2.E.7.e  exact   
5   http://data.europa.eu/esco/skill/15d76317-c71a...  exact   
8   http://data.europa.eu/esco/skill/7111b95d-0ce3...  exact   
9   http://data.europa.eu/esco/skill/4ca91852-c534...  exact   
10  http://data.europa.eu/esco/skill/51586df8-1c46...  exact   
11                         

In [15]:
# Collapse duplicates in exact match dataset
collapsed_exact = skill_exact_match.groupby("skill_clean").agg({
    "framework": lambda x: ", ".join(sorted(set(x))),
    "code": lambda x: ", ".join(sorted(set(x))),
    "method": "first",
    "mapped_label": "first"
}).reset_index()

# Add mentions from the skills table
collapsed_exact = collapsed_exact.merge(table, on="skill_clean", how="left")

# Save to CSV
collapsed_exact.to_csv("skill_exact_collapsed.csv", index=False)

print("Collapsed exact match dataset saved to skill_exact_collapsed.csv")
print(collapsed_exact.head(10))


Collapsed exact match dataset saved to skill_exact_collapsed.csv
           skill_clean   framework  \
0          3d modeling        ESCO   
1           accounting        ESCO   
2   accounting methods        ESCO   
3     active directory        ESCO   
4      active learning        ONET   
5     active listening  ESCO, ONET   
6    actuarial science        ESCO   
7           acute care        ESCO   
8        adobe acrobat        ONET   
9  adobe after effects        ONET   

                                                code method  \
0  http://data.europa.eu/esco/skill/97965983-0da4...  exact   
1  http://data.europa.eu/esco/skill/ecc18804-a466...  exact   
2  http://data.europa.eu/esco/skill/880c54f9-866e...  exact   
3  http://data.europa.eu/esco/skill/d1e954cd-d290...  exact   
4                                            2.A.2.b  exact   
5  2.A.1.b, http://data.europa.eu/esco/skill/a172...  exact   
6  http://data.europa.eu/esco/skill/c9f4efcd-e16a...  exact   
7  http://da

In [17]:
# Add dictionary of abbreviations before LLM step
extra_skills = {
    "ai": "Artificial Intelligence",
    "ml": "Machine Learning",
    "mlops": "Machine Learning Operations",
    "nlp": "Natural Language Processing",
    "bi": "Business Intelligence",
    "sql": "Structured Query Language",
    "etl": "Extract, Transform and Load data for analysis",
    "aws": "Amazon Web Services",
    "gcp": "Google Cloud Platform",
    "crm": "Customer Relationship Management",
    "gis": "Geographic Information Systems",
    "qa": "Quality Assurance",
    "ui": "User Interface",
    "ux": "User Experience",
    "it": "Information Technology",
    "ios": "Apple's operating system",
    "vm": "Virtual Machine",
    "sap": "Systems, Applications, and Products in data processing",
    "sas": "Statistical Analysis System",
    "pyspark": "Python API for Apache Spark",
    "mis": "Management Information Systems",
    "esg": "Environmental, Social and Governance",
    "devops": "Development and Operations",
    "devsecops": "Development, Security and Operations",
    "hr": "Human Resources",
    "hris": "Human Resources Information System",
    "ops": "Operations",
    "dev": "Development",
    "gnc": "Guidance, Navigation and Control",
    "llm": "Large Language Model",
    "ci cd": "Continuous Integration and Continuous Delivery",
    "ui ux": "User Interface/User Experience",
    "vba": "Visual Basic for Applications",
    "ssrs": "SQL Server Reporting Services",
    "dax": "Data Analysis Expressions",
    "dbt": "Data Build Tool",
    "pl/sql": "Procedural Language/SQL",
    "hvac": "Heating, Ventilation, and Air Conditioning",
    "nltk": "Natural Language Toolkit",
    "jupyter": "jupyter notebook"
}


extra_df = pd.DataFrame([
    {"label": k, "mapped_label": v, "framework": "DICT", "code": k}
    for k, v in extra_skills.items()
])

# Append dictionary entries to skill_pool
skill_pool["label"] = skill_pool["label"].str.lower()  # normalize case
skill_pool = pd.concat([skill_pool, extra_df], ignore_index=True)


In [18]:
from rapidfuzz import process, fuzz
import ollama

MODEL = "qwen2.5:7b"

# Candidate search
def get_candidates(skill):
    results = process.extract(skill, skill_pool["label"].tolist(), scorer=fuzz.WRatio, limit=40)
    seen = []
    out = []
    for label, score, i in results:
        row = skill_pool.iloc[i]
        key = row.get("skill_uri", row.get("onet_code",""))
        if key not in seen:
            seen.append(key)
            out.append((row["framework"], key, label))
    return out[:5]

# Ask LLM
def ask_llm(skill, cands):
    options = "\n".join([f"{n+1}. {c[2]} ({c[0]})" for n,c in enumerate(cands)])
    prompt = f"""Skill from a job posting: {skill}

Candidate skills:
{options}

Does any candidate mean the same skill or tool? Reply ONLY with the number, or NONE if none fit."""
    reply = ollama.chat(model=MODEL,
                        messages=[{"role":"user","content":prompt}],
                        options={"temperature":0})
    answer = reply["message"]["content"].strip()
    if answer.isdigit() and 1 <= int(answer) <= len(cands):
        return cands[int(answer)-1]
    return ("NONE","NONE","NONE")


# Build unmatched dataset from skills table, excluding exact matches
unmatched = table[~table["skill_clean"].isin(skill_exact_match["skill_clean"])]

# Deduplicate just in case
unmatched = unmatched.drop_duplicates(subset=["skill_clean"])


# Run on unmatched skills (top 1000)
results = []
for skill in unmatched["skill_clean"].head(1000):

    # Dictionary shortcut
    if skill.lower() in extra_skills:
        results.append({
            "skill_clean": skill,
            "framework": "DICT",
            "code": skill.lower(),
            "mapped_label": extra_skills[skill.lower()],
            "method": "dict"
        })
        print(skill, "->", extra_skills[skill.lower()])
        continue
        
    cands = get_candidates(skill)
    source, code, label = ask_llm(skill, cands)
    results.append({"skill_clean":skill,"framework":source,"code":code,"mapped_label":label,"method":"llm"})
    print(skill,"->",label)

llm_matched = pd.DataFrame(results)

# Save results to CSV
llm_matched.to_csv("llm_matched_skills.csv", index=False)

print("Results saved to llm_matched_skills.csv")

excel -> ms excel
aws -> Amazon Web Services
azure -> NONE
sales -> NONE
gcp -> Google Cloud Platform
recruiting -> NONE
power bi -> pow
product management -> NONE
ci cd -> Continuous Integration and Continuous Delivery
customer success -> NONE
scheduling -> assist in planning production scheduling
llm -> Large Language Model
llms -> sumtotal systems totallms
ui ux -> User Interface/User Experience
kafka -> apache kafka
mlops -> Machine Learning Operations
prompt engineering -> NONE
management -> NONE
databricks -> NONE
qa testing -> NONE
salesforce -> salesforce software
terraform -> ibm terraform
cloud -> NONE
airflow -> airflow analysis
training -> NONE
security -> NONE
marketing -> NONE
word -> use word processing software
hardworking and dedicated -> NONE
powerpoint -> microsoft office powerpoint
looker -> NONE
oracle -> NONE
redshift -> NONE
attention to detail -> attention to detail in casting processes
operations -> NONE
self motivation -> NONE
leadership -> NONE
coordinating -

In [19]:
# Merge LLM results back to ONET pool by label
llm_matched = llm_matched.merge(
    onet_pool[["onet_code","label"]],
    left_on="mapped_label",
    right_on="label",
    how="left"
)

# Fill in missing codes where framework is ONET
llm_matched["code"] = llm_matched.apply(
    lambda row: row["onet_code"] if row["framework"]=="ONET" and pd.isna(row["code"]) else row["code"],
    axis=1
)

# Drop helper column
llm_matched = llm_matched.drop(columns=["label","onet_code"])

# Merge mentions from skills table into LLM results
llm_matched = llm_matched.merge(table, on="skill_clean", how="left")


# Save updated results back to CSV
llm_matched.to_csv("llm_matched_skills.csv", index=False)

print("Updated LLM results saved to llm_matched_skills.csv")


Updated LLM results saved to llm_matched_skills.csv


In [20]:
# Combine exact + LLM matches
lookup = pd.concat([
    matched[["skill_clean","framework"]].rename(columns={"source":"framework"}).assign(method="exact"), 
    llm_matched
])

# Total unique skills in postings
total_skills = table["skill_clean"].nunique() 

# Linked skills (exclude NONE)
linked_skills = lookup[lookup["framework"] != "NONE"]["skill_clean"].nunique()

# Unmapped skills
none_skills = total_skills - linked_skills

print("Summary of skill mapping:")
print(f"Total unique skills in postings: {total_skills}")
print(f"Linked skills: {linked_skills}")
print(f"Unmapped skills: {none_skills}")
print(f"Percentage linked: {round(linked_skills/total_skills*100,1)}%")

print("\nBreakdown by method:")
print(lookup["method"].value_counts())


Summary of skill mapping:
Total unique skills in postings: 4886
Linked skills: 1103
Unmapped skills: 3783
Percentage linked: 22.6%

Breakdown by method:
method
llm      992
exact    803
dict      14
Name: count, dtype: int64


In [22]:
import pandas as pd

# Load both CSVs
exact = pd.read_csv("skill_exact_collapsed.csv")
llm = pd.read_csv("llm_matched_skills.csv")

# Combine them
all_matches = pd.concat([exact, llm], ignore_index=True)

# Save to a new CSV
all_matches.to_csv("all_skill_matches.csv", index=False)

print("Combined dataset saved to all_skill_matches.csv")
print(all_matches.head(10))


Combined dataset saved to all_skill_matches.csv
           skill_clean   framework  \
0          3d modeling        ESCO   
1           accounting        ESCO   
2   accounting methods        ESCO   
3     active directory        ESCO   
4      active learning        ONET   
5     active listening  ESCO, ONET   
6    actuarial science        ESCO   
7           acute care        ESCO   
8        adobe acrobat        ONET   
9  adobe after effects        ONET   

                                                code method  \
0  http://data.europa.eu/esco/skill/97965983-0da4...  exact   
1  http://data.europa.eu/esco/skill/ecc18804-a466...  exact   
2  http://data.europa.eu/esco/skill/880c54f9-866e...  exact   
3  http://data.europa.eu/esco/skill/d1e954cd-d290...  exact   
4                                            2.A.2.b  exact   
5  2.A.1.b, http://data.europa.eu/esco/skill/a172...  exact   
6  http://data.europa.eu/esco/skill/c9f4efcd-e16a...  exact   
7  http://data.europa.eu/esco